In [ ]:
!pip install -q gradio
!pip install -q scikit-learn
!pip install -q joblib
!pip install -q pandas 
!pip install -q numpy

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [12]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RepeatedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

file_path = 'Concrete_Data_simplified.csv'
data = pd.read_csv(file_path)

X = data[['Cement','BlastFurnaceSlag','FlyAsh','Water','Superplasticizer','CoarseAggregate','FineAggregate','Age']]
y = data['Concretecompressivestrength']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rkf = RepeatedKFold(n_splits=5,n_repeats=5,random_state=42)

# using linear regression
lr_model = LinearRegression()
lr_model.fit(X=X_train,y=y_train)

lr_scores = cross_val_score(estimator=lr_model,X=X_train,y=y_train,cv=rkf,scoring='neg_mean_absolute_error')

print("----LINEAR REGRESSION----")
print(f"Negative MAE scores: {lr_scores}")
print(f"Mean cross-validation: {np.mean(lr_scores):.4f}")
print(f"Standard deviation: {np.std(lr_scores):.4f}")

# using random forest
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X=X_train,y=y_train)

rf_scores = cross_val_score(estimator=rf_regressor,X=X_train,y=y_train,cv=rkf,scoring='neg_mean_absolute_error',n_jobs=-1)

print("----RANDOM FOREST----")
print(f"Negative MAE scores: {rf_scores}")
print(f"Mean cross-validation: {np.mean(rf_scores):.4f}")
print(f"Standard deviation: {np.std(rf_scores):.4f}\n")

# prediction results
y_pred_rf = rf_regressor.predict(X_test)
test_mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Test MAE (RF): {test_mae_rf:.4f}")

y_pred_lr = lr_model.predict(X_test)
test_mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f"Test MAE (LR): {test_mae_lr:.4f}")

joblib.dump(rf_regressor, 'rf_model.joblib')
content = {"MAE":test_mae_rf}
file_path = "data.json"
try:
    with open(file_path, "w") as json_file:
        json.dump(content, json_file)
    print(f"Successfully created {file_path}")
except IOError as e:
    print(f"Error creating file: {e}")

FileNotFoundError: [Errno 2] No such file or directory: 'Concrete_Data_simplified.csv'

In [ ]:
import gradio as gr
from joblib import load
import pandas as pd
import json

MAE_CV = 0

try:
    with open('data.json', 'r') as file:
        # Load the JSON data into a Python dictionary
        data = json.load(file)

    # Now you can work with the data as a normal Python dictionary
    MAE_CV = data["MAE"]

except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file. Check if the JSON is valid.")

# These are the constants
K = 2         
MARGIN = K * MAE_CV

rf_model = load('rf_model.joblib')

# Prediction + Decision Logic
def decision_engine(cement, slag, ash, water, plasticizer, coarse, fine, age, target_strength):
    input_data = pd.DataFrame([[cement, slag, ash, water, plasticizer, coarse, fine, age]],
        columns=['Cement','BlastFurnaceSlag','FlyAsh','Water','Superplasticizer','CoarseAggregate','FineAggregate','Age'])
    prediction = rf_model.predict(input_data)

    # Logic part for the decision engine
    if (prediction - MARGIN) >= target_strength:
        status = "✅ PASS"
        note = "Safe to proceed: The mix design is reliably above the target."
    elif (prediction + MARGIN) <= target_strength:
        status = "❌ FAIL"
        note = "Unsafe: The mix design is reliably below the target."
    else:
        status = "⚠️ BORDERLINE"
        note = "Uncertain: Requires manual lab verification before use."

    # RETURN TWO SEPARATE THINGS:
    # 1st: The Strength Info
    strength_info = f"{round(prediction, 2)} MPa (Margin: ±{MARGIN})"
    # 2nd: The Decision Info
    decision_info = f"Status: {status}\n\nAdvisory: {note}\n\nNote: Decision support only; not a substitute for standard lab testing."
    
    return strength_info, decision_info

# 3. the ui
demo = gr.Interface(
    fn=decision_engine,
    inputs=[
        gr.Slider(100, 600, label="Cement (kg)"),
        gr.Slider(0, 400, label="Blast Furnace Slag (kg)"),
        gr.Slider(0, 300, label="Fly Ash (kg)"),
        gr.Slider(100, 300, label="Water (kg)"),
        gr.Slider(0, 35, label="Superplasticizer (kg)"),
        gr.Slider(700, 1200, label="Coarse Aggregate (kg)"),
        gr.Slider(500, 1000, label="Fine Aggregate (kg)"),
        gr.Slider(1, 365, label="Age (Days)"),
        gr.Number(label="Required Target Strength f'c (MPa)", value=10)
    ],
    outputs=[
        gr.Textbox(label="AI Predicted Compressive Strength"),
        gr.Textbox(label="Decision Support Output"),
        gr.Textbox(label="Decision Support Output", lines=8)
    ],
    title="Concrete Compressive Strength Prediction and Decision-Support System Using Supervised Machine Learning",
    theme=gr.themes.Ocean(primary_hue="blue", neutral_hue="slate"),
    live=True,
    allow_flagging="never"
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b690a2f543061abeb5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
